In [ ]:
# ==========================================
# INSTALL REQUIRED LIBRARIES
# ==========================================

!pip install transformers
!pip install torch torchaudio
!pip install gradio
!pip install openai-whisper
!pip install librosa
!pip install speechbrain
!pip install soundfile
!pip install accelerate
!pip install sentencepiece

In [ ]:
# ==========================================
# MOUNT GOOGLE DRIVE
# ==========================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import os
import librosa
import numpy as np
import whisper
import torch
import gradio as gr

from transformers import pipeline
from collections import deque
import re

In [ ]:
# ==========================================
# LOAD WHISPER MODEL
# ==========================================

# Options:
# tiny
# base
# small
# medium
# large

# medium is best for Colab

whisper_model = whisper.load_model("medium")

print("Whisper loaded.")


# ==========================================
# LOAD TOXICITY MODEL
# ==========================================

toxicity_classifier = pipeline(
    "text-classification",
    model="unitary/toxic-bert"
)

print("Toxicity model loaded.")


# ==========================================
# LOAD EMOTION MODEL
# ==========================================

emotion_classifier = pipeline(
    "audio-classification",
    model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
)

print("Emotion model loaded.")

Whisper loaded.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Toxicity model loaded.


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.dense.bias    | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.output.bias   | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
projector.weight         | MISSING    | 
classifier.weight        | MISSING    | 
classifier.bias          | MISSING    | 
projector.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Emotion model loaded.


In [ ]:
# ==========================================
# SESSION MEMORY
# ==========================================

# Stores temporal conversational patterns

toxicity_history = deque(maxlen=20)
emotion_history = deque(maxlen=20)
rage_history = deque(maxlen=20)

In [ ]:
# ==========================================
# CUSTOM CONTEXT FEATURE EXTRACTION
# ==========================================

def extract_context_features(text):

    text = text.lower()

    features = {}

    # ======================================
    # PROFANITY DENSITY
    # ======================================

    toxic_words = [

        "idiot",
        "trash",
        "stupid",
        "moron",
        "loser",
        "ez",
        "dogwater",
        "kill yourself",
        "noob",
        "niggar",
        "bot",
        "fuck shit!"
    ]

    word_count = max(len(text.split()), 1)

    toxic_count = sum(
        word in text
        for word in toxic_words
    )

    features["profanity_density"] = round(
        toxic_count / word_count, 3
    )

    # ======================================
    # PERSONAL ATTACK DETECTION
    # ======================================

    attack_patterns = [

        r"you are",
        r"you're",
        r"ur ",
        r"u are"
    ]

    attack_score = 0

    for pattern in attack_patterns:

        if re.search(pattern, text):
            attack_score += 1

    features["personal_attack"] = (
        min(attack_score * 0.3, 1.0)
    )

    # ======================================
    # REPETITION AGGRESSION
    # ======================================

    repeated_words = re.findall(
        r'\b(\w+)\b(?:\s+\1\b)+',
        text
    )

    features["repetition_aggression"] = (
        min(len(repeated_words) * 0.2, 1.0)
    )

    # ======================================
    # ALL CAPS AGGRESSION
    # ======================================

    caps_words = [

        word for word in text.split()
        if word.isupper()
    ]

    features["caps_aggression"] = (
        min(len(caps_words) * 0.1, 1.0)
    )

    return features

# ==========================================
# AUDIO ENERGY ANALYSIS
# ==========================================

# ======================================
# REMOVE SILENCE
# ======================================

def trim_silence(audio_path):

    import soundfile as sf

    audio, sr = librosa.load(audio_path, sr=16000)

    # Remove silence
    trimmed_audio, _ = librosa.effects.trim(
        audio,
        top_db=20
    )

    temp_path = "/content/temp_trimmed.wav"

    sf.write(temp_path, trimmed_audio, sr)

    return temp_path


# ======================================
# NORMALIZE EMOTION LABELS
# ======================================

def normalize_emotion(label):

    label = label.lower()

    if "angry" in label or "anger" in label:
        return "angry"

    elif "happy" in label:
        return "happy"

    elif "neutral" in label:
        return "neutral"

    elif "sad" in label:
        return "sad"

    else:
        return "other"

def analyze_voice_energy(audio_path):
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)
    # RMS energy
    rms = np.mean(librosa.feature.rms(y=audio))
    # Normalize score
    shouting_score = min(rms * 20, 1.0)
    return shouting_score

In [ ]:
# ==========================================
# TEMPORAL ESCALATION ANALYSIS
# ==========================================

def compute_escalation():

    if len(toxicity_history) < 3:
        return 0.0

    recent = list(toxicity_history)

    # ======================================
    # TOXICITY TREND
    # ======================================

    trend = recent[-1] - recent[0]

    # ======================================
    # RECENT MOVING AVERAGE
    # ======================================

    avg_recent = np.mean(recent[-5:])

    # ======================================
    # FINAL ESCALATION SCORE
    # ======================================

    escalation = (
        trend * 0.6 +
        avg_recent * 0.4
    )

    return min(max(escalation, 0), 1.0)

# ==========================================
# RAGE DETECTION
# ==========================================

def detect_rage(emotion_label, shouting_score):

    rage_score = 0

    # Angry emotion
    if emotion_label.lower() == "angry":
        rage_score += 0.5

    # Loud aggressive voice
    if shouting_score > 0.6:
        rage_score += 0.5

    return min(rage_score, 1.0)

In [ ]:
# ==========================================
# MAIN TOXICITY DETECTION PIPELINE
# ==========================================

# ==========================================
# ADAPTIVE CONTEXT-AWARE FUSION ENGINE
# ==========================================

def adaptive_fusion(

    toxicity_score,
    shouting_score,
    rage_score,
    escalation_score,
    emotion_confidence,
    context_features
):

    # ======================================
    # BASE DYNAMIC WEIGHTS
    # ======================================

    w_text = 0.50
    w_shouting = 0.15
    w_rage = 0.15
    w_escalation = 0.10
    w_context = 0.10

    # ======================================
    # ADAPTIVE WEIGHTING
    # ======================================

    # Increase rage importance
    # during aggressive shouting

    if shouting_score > 0.6:

        w_rage += 0.10
        w_text -= 0.05

    # Increase escalation sensitivity

    if escalation_score > 0.5:

        w_escalation += 0.10
        w_text -= 0.05

    # Reduce emotion importance
    # if confidence is weak

    if emotion_confidence < 0.4:

        w_rage -= 0.05
        w_text += 0.05

    # ======================================
    # CONTEXTUAL AGGRESSION SCORE
    # ======================================

    context_score = (
        context_features["profanity_density"] * 0.4 +
        context_features["personal_attack"] * 0.3 +
        context_features["repetition_aggression"] * 0.2 +
        context_features["caps_aggression"] * 0.1
    )

    # ======================================
    # FINAL MULTIMODAL FUSION
    # ======================================

    final_score = (
        toxicity_score * w_text +
        shouting_score * w_shouting +
        rage_score * w_rage +
        escalation_score * w_escalation +
        context_score * w_context
    )

    return min(final_score, 1.0)

def detect_toxicity(audio_path):

    # ======================================
    # 1. SPEECH RECOGNITION
    # ======================================

    whisper_result = whisper_model.transcribe(
      audio_path,
      fp16=False,
      language="en",
      temperature=0.2
    )

    text = whisper_result["text"]

    # ======================================
    # CONTEXTUAL FEATURE EXTRACTION
    # ======================================

    context_features = extract_context_features(text)

    # --------------------------------------
    # HANDLE EMPTY TRANSCRIPTION
    # --------------------------------------

    if len(text.strip()) < 3:

        return {
            "Transcribed Speech": "No speech detected",
            "Toxicity Label": "UNKNOWN",
            "Text Toxicity Score": 0.0,
            "Emotion": "neutral",
            "Emotion Confidence": 0.0,
            "Shouting Score": 0.0,
            "Rage Score": 0.0,
            "Escalation Score": 0.0,
            "Final Toxicity Score": 0.0,
            "Verdict": "NO SPEECH DETECTED"
        }

    # ======================================
    # 2. SMART TOXICITY ANALYSIS
    # ======================================

    toxicity_result = toxicity_classifier(text)[0]
    raw_score = float(toxicity_result["score"])
    label = toxicity_result["label"].lower()

    # --------------------------------------
    # BASE TOXICITY SCORE
    # --------------------------------------

    toxicity_score = 0.0

    # Convert labels into weighted scores

    if "hate" in label:
        toxicity_score += 0.90

    elif "toxic" in label:
        toxicity_score += 0.75

    elif "insult" in label:
        toxicity_score += 0.65

    elif "obscene" in label:
        toxicity_score += 0.60

    else:
        toxicity_score += 0.20


    # Multiply by model confidence
    toxicity_score *= raw_score


    # --------------------------------------
    # BOOST SCORE USING TEXT ANALYSIS
    # --------------------------------------

    gaming_toxic_words = [
        "trash",
        "idiot",
        "stupid",
        "ez",
        "loser",
        "dogwater",
        "kill yourself",
        "noob",
        "moron"
    ]

    text_lower = text.lower()

    word_boost = 0

    for word in gaming_toxic_words:

        if word in text_lower:
            word_boost += 0.15

    toxicity_score += word_boost


    # Cap score
    toxicity_score = min(toxicity_score, 1.0)


    # --------------------------------------
    # DYNAMIC TOXICITY LABEL
    # --------------------------------------

    if toxicity_score >= 0.75:
        toxicity_label = "HIGHLY TOXIC"
    elif toxicity_score >= 0.45:
        toxicity_label = "MODERATELY TOXIC"
    else:
        toxicity_label = "NON-TOXIC"

    # ======================================
    # 3. SMART EMOTION DETECTION
    # ======================================

    # Remove silence first
    trimmed_audio = trim_silence(audio_path)

    # Predict emotions
    emotion_results = emotion_classifier(trimmed_audio)

    # Default values
    emotion_label = "neutral"
    emotion_score = 0.0

    # Store emotion scores
    emotion_scores = {}

    valid_emotions = [
      "angry",
      "neutral",
      "happy"
    ]

    for emotion in emotion_results:

      label = normalize_emotion(
          emotion["label"]
      )

      if label not in valid_emotions:
          continue

      score = float(emotion["score"])
      emotion_scores[label] = score

    # --------------------------------------
    # SMART DECISION LOGIC
    # --------------------------------------

    angry_score = emotion_scores.get("angry", 0)
    neutral_score = emotion_scores.get("neutral", 0)
    happy_score = emotion_scores.get("happy", 0)


    # Only classify angry if VERY confident

    # ======================================
    # SMART EMOTION CONFIDENCE SCALING
    # ======================================

    top_emotion = max(
        emotion_scores,
        key=emotion_scores.get
    )

    top_score = emotion_scores[top_emotion]

    # --------------------------------------
    # AMPLIFY LOW PROBABILITIES
    # --------------------------------------

    # Convert weak probabilities
    # into stronger confidence values

    scaled_score = min(top_score * 5.0, 1.0)

    # --------------------------------------
    # FINAL EMOTION DECISION
    # --------------------------------------

    if scaled_score >= 0.65:
        emotion_label = top_emotion
        emotion_score = scaled_score

    else:
        emotion_label = "neutral"
        emotion_score = scaled_score * 0.5


    # ======================================
    # 4. SHOUTING DETECTION
    # ======================================

    shouting_score = analyze_voice_energy(audio_path)


    # ======================================
    # 5. RAGE DETECTION
    # ======================================

    rage_score = detect_rage(
        emotion_label,
        shouting_score
    )

    # ======================================
    # 6. TEMPORAL ESCALATION MEMORY
    # ======================================

    toxicity_history.append(toxicity_score)
    emotion_history.append(emotion_score)
    rage_history.append(rage_score)

    escalation_score = compute_escalation()

    # ======================================
    # ADAPTIVE CONTEXT-AWARE FUSION
    # ======================================

    final_score = adaptive_fusion(

        toxicity_score,
        shouting_score,
        rage_score,
        escalation_score,
        emotion_score,
        context_features
    )


    # ======================================
    # 8. FINAL DECISION
    # ======================================

    if final_score >= 0.70:
        verdict = "HIGHLY TOXIC 🔴"

    elif final_score >= 0.40:
        verdict = "MODERATELY TOXIC 🟠"

    else:
        verdict = "NON-TOXIC 🟢"


    # ======================================
    # 9. RETURN RESULTS
    # ======================================

    return {
        "Transcribed Speech": text,
        "Toxicity Label": toxicity_label,
        "Text Toxicity Score": round(float(toxicity_score), 3),
        "Emotion": emotion_label,
        "Emotion Confidence": round(float(emotion_score), 3),
        "Shouting Score": round(float(shouting_score), 3),
        "Rage Score": round(float(rage_score), 3),
        "Escalation Score": round(float(escalation_score), 3),
        "Context Features": context_features,
        "Final Toxicity Score": round(float(final_score), 3),
        "Verdict": verdict
    }

In [ ]:
# ==========================================
# BUILD REAL-TIME GRADIO INTERFACE
# ==========================================

interface = gr.Interface(

    fn=detect_toxicity,

    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎤 Speak Into Microphone"
    ),

    outputs="json",

    live=True,

    title="🎮 Real-Time Gaming Voice Toxicity Detector",

    description="""
    This AI system analyzes:

    Toxic speech
    -Angry tone
    -Shouting
    -Rage behavior
    -Toxic escalation

    Built using:
    - Whisper
    - Toxic-BERT
    - Emotion AI
    - Audio Signal Processing
    """
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e67f43c0708ed7c8df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
